# Tokenizer Tutorial (`CharTokenizer`, `BPETokenizer`, `ByteBPETokenizer`)

This notebook explains how tokenization works in `src/nano_llm/tokenizer.py`.

You will learn:
- how character tokenization works
- how BPE learns merges
- what `_merge_pair()` does step by step
- how standard BPE differs from word-boundary-aware BPE
- how byte-level BPE behaves on Unicode text
- how encode/decode differs between char and BPE variants
- how checkpoint tokenizer state works

In [1]:
from pathlib import Path
import sys

# Make src/ importable whether notebook cwd is repo root or docs/.
candidates = [Path.cwd(), Path.cwd().parent]
repo_root = next((p for p in candidates if (p / "src" / "nano_llm").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing src/nano_llm")

sys.path.insert(0, str(repo_root / "src"))

from nano_llm.tokenizer import (
    CharTokenizer,
    BPETokenizer,
    ByteBPETokenizer,
    build_tokenizer_from_text,
    tokenizer_from_state,
)

## 1) Character tokenizer basics

`CharTokenizer` maps each character to one id.
So the sequence length equals the number of characters.

In [2]:
text = "ROMEO: Hello\n"
char_tok = CharTokenizer.from_text(text, add_special=False)

ids = char_tok.encode(text)
decoded = char_tok.decode(ids)

print("vocab size:", char_tok.vocab_size)
print("ids:", ids)
print("roundtrip:", decoded)
print("length chars:", len(text), "length ids:", len(ids))

vocab size: 11
ids: [7, 6, 5, 3, 6, 2, 1, 4, 8, 9, 9, 10, 0]
roundtrip: ROMEO: Hello

length chars: 13 length ids: 13


## 2) `_merge_pair()` (core BPE primitive)

This function applies one merge rule to a token list.

Given pair `(a, b)`:
- if adjacent tokens match `a, b`, it replaces them with `a+b`
- otherwise it copies the token as-is
- it skips overlaps in a single pass

In [3]:
tokens = ["t", "h", "e", " ", "t", "h", "e"]
merged = BPETokenizer._merge_pair(tokens, ("t", "h"))
print("before:", tokens)
print("after :", merged)

tokens_overlap = ["a", "a", "a"]
merged_overlap = BPETokenizer._merge_pair(tokens_overlap, ("a", "a"))
print("overlap example:", tokens_overlap, "->", merged_overlap)

before: ['t', 'h', 'e', ' ', 't', 'h', 'e']
after : ['th', 'e', ' ', 'th', 'e']
overlap example: ['a', 'a', 'a'] -> ['aa', 'a']


## 3) Learn a tiny BPE tokenizer from text

`BPETokenizer.from_text()` starts from characters and repeatedly merges the most frequent adjacent pair until `vocab_size` is reached (or no useful merge remains).

By default, merges can include whitespace, so long phrase-like tokens can appear on very small corpora.

In [4]:
train_text = "to be or not to be, that is the question. to be or not."
bpe_tok = BPETokenizer.from_text(train_text, vocab_size=80)

print("vocab size:", bpe_tok.vocab_size)
print("num merges:", len(bpe_tok.merges))
print("first 10 merges:", bpe_tok.merges[:10])

vocab size: 28
num merges: 12
first 10 merges: [(' ', 't'), ('o', ' '), ('o ', 'b'), ('o b', 'e'), ('o be', ' '), ('o be ', 'o'), ('o be o', 'r'), ('o be or', ' '), ('o be or ', 'n'), ('o be or n', 'o')]


In [5]:
sample = "to be or not to be"
bpe_ids = bpe_tok.encode(sample)
bpe_decoded = bpe_tok.decode(bpe_ids)

print("sample:", sample)
print("ids:", bpe_ids)
print("roundtrip:", bpe_decoded)
print("char len:", len(sample), "bpe len:", len(bpe_ids))

sample: to be or not to be
ids: [26, 22, 2, 15]
roundtrip: to be or not to be
char len: 18 bpe len: 4


## 4) Char vs standard BPE vs word-boundary-aware BPE

- Standard BPE may merge across spaces (fewer tokens, but can become phrase-heavy on tiny data).
- Word-boundary-aware BPE prevents merges that include whitespace (more stable word-level chunks).
- Char tokenizer is the baseline (one token per character).

In [6]:
sample = "ROMEO: to be or not to be"
train_for_bpe = sample * 40

char_tok2 = build_tokenizer_from_text(train_for_bpe, tokenizer_type="char")
bpe_std = build_tokenizer_from_text(
    train_for_bpe,
    tokenizer_type="bpe",
    bpe_vocab_size=120,
    bpe_word_boundary_aware=False,
)
bpe_wb = build_tokenizer_from_text(
    train_for_bpe,
    tokenizer_type="bpe",
    bpe_vocab_size=120,
    bpe_word_boundary_aware=True,
)

char_ids = char_tok2.encode(sample)
std_ids = bpe_std.encode(sample)
wb_ids = bpe_wb.encode(sample)

print("char token count:", len(char_ids))
print("std bpe token count:", len(std_ids))
print("wb bpe token count :", len(wb_ids))
print()
print("std bpe has merged space token:", any((" " in t) and (len(t) > 1) for t in bpe_std.vocab))
print("wb bpe has merged space token :", any((" " in t) and (len(t) > 1) for t in bpe_wb.vocab))
print()
print("char decode ok:", char_tok2.decode(char_ids) == sample)
print("std  decode ok:", bpe_std.decode(std_ids) == sample)
print("wb   decode ok:", bpe_wb.decode(wb_ids) == sample)

char token count: 25
std bpe token count: 1
wb bpe token count : 13

std bpe has merged space token: True
wb bpe has merged space token : False

char decode ok: True
std  decode ok: True
wb   decode ok: True


## 5) Checkpoint-compatible tokenizer state

Training saves tokenizer metadata in checkpoint (`tokenizer_state`).
You can roundtrip with `to_state()` and `tokenizer_from_state()`.

In [7]:
state = bpe_tok.to_state()
restored = tokenizer_from_state(state)

text = "to be"
print("type:", state["type"])
print("same encoding:", restored.encode(text) == bpe_tok.encode(text))
print("same decoding:", restored.decode(restored.encode(text)) == text)

type: bpe
same encoding: True
same decoding: True


## 6) How to use in training

Use char tokenizer (default):

```bash
python scripts/train.py
```

Use standard BPE tokenizer:

```bash
python scripts/train.py --tokenizer-type bpe --bpe-vocab-size 256
```

Use word-boundary-aware BPE tokenizer:

```bash
python scripts/train.py --tokenizer-type bpe --bpe-vocab-size 256 --bpe-word-boundary-aware
```

Use byte-level BPE tokenizer:

```bash
python scripts/train.py --tokenizer-type bpe_byte --bpe-vocab-size 256
```

## 7) `bpe_word_boundary_aware` in detail

When `bpe_word_boundary_aware=True`, BPE will **not** merge pairs if either token contains whitespace.

Practical effect:
- prevents cross-word phrase tokens like `"o be or n"`
- keeps merges inside words (or punctuation-neighbor tokens without spaces)
- usually gives more interpretable subword vocab on small datasets

Tradeoff:
- token count may be a bit higher than standard BPE
- but behavior is often more stable and less phrase-overfitted on tiny corpora

In [8]:
toy = "to be or not to be. to be or not."

std = BPETokenizer.from_text(toy, vocab_size=80, word_boundary_aware=False)
wb = BPETokenizer.from_text(toy, vocab_size=80, word_boundary_aware=True)

print("standard BPE first 12 merges:")
print(std.merges[:12])
print()
print("word-boundary-aware BPE first 12 merges:")
print(wb.merges[:12])
print()

std_has_cross_word = any((" " in (a + b)) for a, b in std.merges)
wb_has_cross_word = any((" " in (a + b)) for a, b in wb.merges)
print("standard has cross-word merges:", std_has_cross_word)
print("wb-aware has cross-word merges:", wb_has_cross_word)

sample = "to be or not to be"
print("\nsample:", sample)
print("std token count:", len(std.encode(sample)))
print("wb token count :", len(wb.encode(sample)))

standard BPE first 12 merges:
[('t', 'o'), ('to', ' '), ('to ', 'b'), ('to b', 'e'), ('to be', ' '), ('to be ', 'o'), ('to be o', 'r'), ('to be or', ' '), ('to be or ', 'n'), ('to be or n', 'o'), ('to be or no', 't')]

word-boundary-aware BPE first 12 merges:
[('t', 'o'), ('b', 'e'), ('o', 'r'), ('n', 'o'), ('no', 't')]

standard has cross-word merges: True
wb-aware has cross-word merges: False

sample: to be or not to be
std token count: 3
wb token count : 11


### Rule of thumb

- Use `bpe_word_boundary_aware=False` if you care most about compression (fewer tokens).
- Use `bpe_word_boundary_aware=True` if you care more about cleaner word-level subwords and robustness on small corpora.

For Tiny Shakespeare experiments, start with:

```bash
python scripts/train.py --tokenizer-type bpe --bpe-vocab-size 256 --bpe-word-boundary-aware
```

## 8) Byte-level BPE quick demo

`ByteBPETokenizer` first converts text to UTF-8 bytes and applies BPE on byte symbols.

Why this is useful:
- handles any Unicode input without unknown characters
- robust for mixed scripts/emojis/symbol-heavy text
- closer to tokenizer design used by many modern LLMs

In [9]:
unicode_text = "Cafe\u0301 | 你好 | 😀 | مرحبا"

byte_bpe = ByteBPETokenizer.from_text(unicode_text * 20, vocab_size=256, word_boundary_aware=True)
char_tok3 = CharTokenizer.from_text(unicode_text, add_special=False)

byte_ids = byte_bpe.encode(unicode_text)
char_ids = char_tok3.encode(unicode_text)

print("input:", unicode_text)
print("char token count:", len(char_ids))
print("byte-bpe token count:", len(byte_ids))
print("byte-bpe roundtrip:", byte_bpe.decode(byte_ids) == unicode_text)
print("tokenizer state type:", byte_bpe.to_state()["type"])

input: Café | 你好 | 😀 | مرحبا
char token count: 22
byte-bpe token count: 17
byte-bpe roundtrip: True
tokenizer state type: bpe_byte


### Byte-level BPE tip

When your dataset contains lots of non-ASCII or mixed-language text, `bpe_byte` is usually a safer default than char-seeded BPE.

## 9) Hugging Face byte-level BPE (`hf_bpe_byte`)

This mode uses the modern `tokenizers` pipeline under the hood (BPE model + ByteLevel pre-tokenizer/decoder).

Why use it:
- closer to real production LLM tokenization workflows
- robust Unicode handling
- easy interoperability with `tokenizer.json`-style tooling

In [10]:
# Requires: pip install tokenizers
hf_tok = build_tokenizer_from_text(
    "Cafe\u0301 | 你好 | 😀 | مرحبا " * 10,
    tokenizer_type="hf_bpe_byte",
    bpe_vocab_size=256,
    bpe_word_boundary_aware=False,
)

sample = "Cafe\u0301 | 你好 | 😀 | مرحبا"
ids = hf_tok.encode(sample)
print("hf_bpe_byte token count:", len(ids))
print("hf_bpe_byte roundtrip:", hf_tok.decode(ids) == sample)
print("state type:", hf_tok.to_state()["type"])

hf_bpe_byte token count: 8
hf_bpe_byte roundtrip: True
state type: hf_bpe_byte


Use in training:

```bash
python scripts/train.py --tokenizer-type hf_bpe_byte --bpe-vocab-size 256
```